In [ ]:
#!/usr/bin/env python3

# ==========================================================
# INFERENCE FOR CONTOUR1 / CONTOUR2 / CONTOUR3
# ==========================================================

import torch
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt

from contour1_model import (
    ContourModel1_CNN,
    resample,
    normalize_triplet,
    DEVICE,
    N_POINTS
)

from contour2_model import (
    ContourModel2_TCN,
    encode_n
)

from contour3_model import (
    ResidualContourModel
)

# ==========================================================
# ALPHAS (DEV-TUNED)
# ==========================================================

ALPHA_C1 = 0.30
ALPHA_C2 = 0.40
ALPHA_C3 = 0.10

# ==========================================================
# FILE PATHS
# ==========================================================

PREV_FILE = "prev.mat"
FUT_FILE = "fut.mat"

MODEL_C1 = "SA_contour1_model.pth"
MODEL_C2 = "SA_contour2_model.pth"
MODEL_C3 = "SA_contour3_model.pth"

# ==========================================================
# LOAD CONTOURS
# ==========================================================

def load_three_contours(mat_file):

    data = sio.loadmat(mat_file, squeeze_me=True)

    arrays = []

    for k, v in data.items():

        if k.startswith("__"):
            continue

        if isinstance(v, np.ndarray):

            if v.ndim == 2:

                arrays.append(v)

    if len(arrays) < 3:

        raise ValueError(
            f"{mat_file} does not contain 3 contour arrays."
        )

    return arrays[0], arrays[1], arrays[2]

# ==========================================================
# PREPROCESS
# ==========================================================

def prepare_pair(prev_raw, fut_raw):

    prev_pts = resample(prev_raw, N_POINTS)
    fut_pts  = resample(fut_raw, N_POINTS)

    if prev_pts is None or fut_pts is None:

        raise ValueError(
            "Resampling failed."
        )

    dummy_gt = np.zeros_like(prev_pts)

    prev_norm, fut_norm, _, mean, std = normalize_triplet(
        prev_pts,
        fut_pts,
        dummy_gt
    )

    return (
        prev_norm,
        fut_norm,
        mean,
        std
    )

# ==========================================================
# LOAD MODELS
# ==========================================================

print("\nLoading models...")

model_c1 = ContourModel1_CNN().to(DEVICE)
model_c1.load_state_dict(
    torch.load(MODEL_C1, map_location=DEVICE)
)
model_c1.eval()

model_c2 = ContourModel2_TCN().to(DEVICE)
model_c2.load_state_dict(
    torch.load(MODEL_C2, map_location=DEVICE)
)
model_c2.eval()

model_c3 = ResidualContourModel().to(DEVICE)
model_c3.load_state_dict(
    torch.load(MODEL_C3, map_location=DEVICE)
)
model_c3.eval()

print("Models loaded successfully.")

# ==========================================================
# LOAD INPUT
# ==========================================================

print("\nLoading input contours...")

prev_c1, prev_c2, prev_c3 = load_three_contours(
    PREV_FILE
)

fut_c1, fut_c2, fut_c3 = load_three_contours(
    FUT_FILE
)

# ==========================================================
# PREPARE DATA
# ==========================================================

prev1, fut1, mean1, std1 = prepare_pair(
    prev_c1,
    fut_c1
)

prev2, fut2, mean2, std2 = prepare_pair(
    prev_c2,
    fut_c2
)

prev3, fut3, mean3, std3 = prepare_pair(
    prev_c3,
    fut_c3
)

# ==========================================================
# CONVERT TO TENSORS
# ==========================================================

prev1_t = torch.tensor(
    prev1,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

fut1_t = torch.tensor(
    fut1,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

prev2_t = torch.tensor(
    prev2,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

fut2_t = torch.tensor(
    fut2,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

prev3_t = torch.tensor(
    prev3,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

fut3_t = torch.tensor(
    fut3,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

# ==========================================================
# INFERENCE
# ==========================================================

with torch.no_grad():

    # ------------------------------------------------------
    # CONTOUR 1
    # ------------------------------------------------------

    residual1 = model_c1(
        prev1_t,
        fut1_t
    )

    linear1 = (
        prev1_t + fut1_t
    ) / 2.0

    pred1 = (
        linear1
        +
        ALPHA_C1 * residual1
    )

    pred1 = pred1.squeeze(0).cpu().numpy()

    pred1 = pred1 * std1 + mean1

    # ------------------------------------------------------
    # CONTOUR 2
    # ------------------------------------------------------

    motion2 = fut2_t - prev2_t

    n_enc = encode_n(1)

    n_enc = torch.tensor(
        n_enc,
        dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    residual2 = model_c2(
        prev2_t,
        fut2_t,
        motion2,
        n_enc
    )

    linear2 = (
        prev2_t + fut2_t
    ) / 2.0

    pred2 = (
        linear2
        +
        ALPHA_C2 * residual2
    )

    pred2 = pred2.squeeze(0).cpu().numpy()

    pred2 = pred2 * std2 + mean2

    # ------------------------------------------------------
    # CONTOUR 3
    # ------------------------------------------------------

    residual3 = model_c3(
        prev3_t,
        fut3_t
    )

    linear3 = (
        prev3_t + fut3_t
    ) / 2.0

    pred3 = (
        linear3
        +
        ALPHA_C3 * residual3
    )

    pred3 = pred3.squeeze(0).cpu().numpy()

    pred3 = pred3 * std3 + mean3

# ==========================================================
# SAVE MAT FILES
# ==========================================================

sio.savemat(
    "predicted_contour1.mat",
    {"contour1": pred1}
)

sio.savemat(
    "predicted_contour2.mat",
    {"contour2": pred2}
)

sio.savemat(
    "predicted_contour3.mat",
    {"contour3": pred3}
)

print("Saved MAT files")

# ==========================================================
# SAVE PNG VISUALIZATIONS
# ==========================================================

def save_contour_plot(contour, title, filename):

    plt.figure(figsize=(6,6))

    plt.plot(
        contour[:,0],
        contour[:,1],
        linewidth=2
    )

    plt.gca().invert_yaxis()

    plt.axis("equal")

    plt.title(title)

    plt.tight_layout()

    plt.savefig(filename)

    plt.close()

save_contour_plot(
    pred1,
    "Predicted Contour 1",
    "contour1_prediction.png"
)

save_contour_plot(
    pred2,
    "Predicted Contour 2",
    "contour2_prediction.png"
)

save_contour_plot(
    pred3,
    "Predicted Contour 3",
    "contour3_prediction.png"
)

print("Saved PNG visualizations")